# H1.2 — Ablación y evaluación con semillas nuevas de DDPG-LQR e ISMC

**Propósito:** comprobar, sin reentrenar, la aportación del *swing-up* aprendido por DDPG frente al control local de LQR/ISMC. Este experimento conserva la dinámica y las ganancias H1/H1.1; no ajusta parámetros con las semillas finales.

- **Cinco arquitecturas:** DDPG, LQR solo, ISMC solo, DDPG-LQR, DDPG-ISMC.
- **Políticas retenidas:** siete checkpoints de N6.1 no usados en H1.1, pero ya considerados en H1; control local puro no depende del checkpoint y se evalúa una sola vez por semilla/escenario.
- **Semillas de evaluación nuevas:** 8500 y 8501 (sin selección de configuración); evaluación desde abajo durante 120 s.
- **Escenarios:** sin perturbación, +5 N durante 0.2 s, −5 N durante 0.2 s, +10 N durante 0.2 s y +5 N durante 1 s.
- **Ablación adicional cerca de la vertical** para validar funcionamiento local, sin confundirla con el *swing-up*.

**Métricas:** estabilidad estricta continua máxima, ocupación estricta, entrada vertical, activación local, tiempo de recuperación, salidas del modo local, actuación y supervivencia. Los episodios se guardan individualmente en Drive y se reanudan al volver a ejecutar.

## 1. Dependencias y persistencia

In [ ]:
%pip -q install 'stable-baselines3>=2.3,<3' 'gymnasium>=0.29,<2' 'scipy>=1.10' 'pandas>=2' 'matplotlib>=3'

In [ ]:
from pathlib import Path
import io, os, json, math, hashlib, zipfile, time, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are
from stable_baselines3 import DDPG
from stable_baselines3.common.buffers import ReplayBuffer
from google.colab import files, drive

drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/H12_VALIDACION_HIBRIDOS')
OUT.mkdir(parents=True,exist_ok=True)
assert str(OUT.resolve()).startswith('/content/drive/MyDrive/')
ping=OUT/'prueba_escritura.txt';ping.write_text('ok',encoding='utf8');assert ping.read_text()=='ok'
print('Guardado verificado:',OUT)

## 2. Verificación de checkpoints N6.1 (sin reentrenamiento)

In [ ]:
PACKAGE_NAME='Hibridos_DDPG_N61_MODELOS_SELECCIONADOS.zip'
package=OUT/PACKAGE_NAME
if not package.exists():
    old=Path('/content/drive/MyDrive/H1_HIBRIDOS_DDPG_SIN_REENTRENAR')/PACKAGE_NAME
    if old.exists():
        import shutil;shutil.copy2(old,package);print('Modelos reutilizados desde H1')
if not package.exists():
    print('Selecciona exactamente:',PACKAGE_NAME)
    uploaded=files.upload()
    assert PACKAGE_NAME in uploaded, 'Se necesita el paquete de modelos seleccionados'
    package.write_bytes(uploaded[PACKAGE_NAME])
assert zipfile.ZipFile(package).testzip() is None
with zipfile.ZipFile(package) as z:
    manifest=json.loads(z.read('MANIFEST.json'))
    assert len(manifest)==10
    for item in manifest:
        content=z.read(item['archivo'])
        assert hashlib.sha256(content).hexdigest()==item['sha256']
        target=OUT/item['archivo'];target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists() or hashlib.sha256(target.read_bytes()).hexdigest()!=item['sha256']:
            target.write_bytes(content)
    selected=pd.read_csv(io.BytesIO(z.read('seleccion_validacion.csv')))
assert len(selected)==10
models={}
for item in manifest:
    key=(int(item['seed']),item['tratamiento'])
    m=DDPG.load(str(OUT/item['archivo']),device='cpu',
        custom_objects={'replay_buffer_class':ReplayBuffer,'replay_buffer_kwargs':{}})
    assert m.observation_space.shape==(4,) and m.action_space.shape==(1,)
    assert m.num_timesteps==int(item['checkpoint'])
    models[key]=m
print('Modelos cargados y verificados:',len(models))
print(selected[['train_seed','tratamiento','checkpoint','estabilidad_max_s']].to_string(index=False))

## 3. Dinámica física original y controladores H1 (sin alterar ecuaciones ni ganancias)

In [ ]:
from pathlib import Path
import gymnasium as gym
import numpy as np
import math
ORIGINAL_ENV='"""\nEntorno Gymnasium: péndulo invertido tipo carro–poste con acción continua.\n\nEstado:\n    s_t = [x_t, x_dot_t, theta_t, theta_dot_t]\n\nAcción:\n    a_t = u(t), fuerza continua aplicada al carro.\n\nConvención angular:\n    theta = 0  -> péndulo vertical hacia arriba.\n    theta = pi -> péndulo hacia abajo.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Optional\n\nimport gymnasium as gym\nfrom gymnasium import spaces\nimport numpy as np\n\n\nclass CartPoleContinuousEnv(gym.Env):\n    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}\n\n    def __init__(\n        self,\n        render_mode: Optional[str] = None,\n        max_episode_steps: int = 1000,\n        force_max: float = 10.0,\n        dt: float = 0.02,\n        x_threshold: float = 2.4,\n        noise_std: float = 0.0,\n        viscous_friction_cart: float = 0.0,\n        viscous_friction_pole: float = 0.0,\n        external_impulse: float = 0.0,\n        impulse_step: Optional[int] = None,\n        random_initial_state: bool = True,\n        start_near_downward: bool = True,\n    ):\n        super().__init__()\n\n        self.gravity = 9.81\n        self.masscart = 1.0\n        self.masspole = 0.1\n        self.total_mass = self.masscart + self.masspole\n        self.length = 0.5\n        self.polemass_length = self.masspole * self.length\n\n        self.force_max = float(force_max)\n        self.dt = float(dt)\n        self.x_threshold = float(x_threshold)\n        self.max_episode_steps = int(max_episode_steps)\n\n        self.noise_std = float(noise_std)\n        self.viscous_friction_cart = float(viscous_friction_cart)\n        self.viscous_friction_pole = float(viscous_friction_pole)\n        self.external_impulse = float(external_impulse)\n        self.impulse_step = impulse_step\n\n        self.random_initial_state = random_initial_state\n        self.start_near_downward = start_near_downward\n\n        self.render_mode = render_mode\n        self.screen = None\n        self.clock = None\n        self.screen_width = 800\n        self.screen_height = 500\n\n        self.action_space = spaces.Box(\n            low=np.array([-self.force_max], dtype=np.float32),\n            high=np.array([self.force_max], dtype=np.float32),\n            dtype=np.float32,\n        )\n\n        high = np.array(\n            [\n                self.x_threshold * 2.0,\n                np.finfo(np.float32).max,\n                math.pi,\n                np.finfo(np.float32).max,\n            ],\n            dtype=np.float32,\n        )\n        self.observation_space = spaces.Box(-high, high, dtype=np.float32)\n\n        self.state: Optional[np.ndarray] = None\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n    @staticmethod\n    def angle_normalize(theta: float) -> float:\n        return ((theta + math.pi) % (2.0 * math.pi)) - math.pi\n\n    def _get_obs(self) -> np.ndarray:\n        assert self.state is not None\n        obs = self.state.copy()\n\n        if self.noise_std > 0:\n            noise = self.np_random.normal(0.0, self.noise_std, size=obs.shape)\n            obs = obs + noise.astype(np.float32)\n\n        obs[2] = self.angle_normalize(float(obs[2]))\n        return obs.astype(np.float32)\n\n    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):\n        super().reset(seed=seed)\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n        if self.random_initial_state:\n            if self.start_near_downward:\n                x = self.np_random.uniform(low=-0.05, high=0.05)\n                x_dot = self.np_random.uniform(low=-0.05, high=0.05)\n                theta = math.pi + self.np_random.uniform(low=-0.10, high=0.10)\n                theta_dot = self.np_random.uniform(low=-0.05, high=0.05)\n            else:\n                x = self.np_random.uniform(low=-0.5, high=0.5)\n                x_dot = self.np_random.uniform(low=-0.5, high=0.5)\n                theta = self.np_random.uniform(low=-math.pi, high=math.pi)\n                theta_dot = self.np_random.uniform(low=-1.0, high=1.0)\n        else:\n            x, x_dot, theta, theta_dot = 0.0, 0.0, math.pi, 0.0\n\n        theta = self.angle_normalize(theta)\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n\n        info = {\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n        }\n        return self._get_obs(), info\n\n    def step(self, action):\n        assert self.state is not None\n\n        x, x_dot, theta, theta_dot = map(float, self.state)\n        force = float(np.clip(action[0], -self.force_max, self.force_max))\n\n        if self.impulse_step is not None and self.steps == self.impulse_step:\n            force += self.external_impulse\n\n        force -= self.viscous_friction_cart * x_dot\n\n        costheta = math.cos(theta)\n        sintheta = math.sin(theta)\n\n        temp = (\n            force + self.polemass_length * theta_dot**2 * sintheta\n        ) / self.total_mass\n\n        theta_acc = (\n            self.gravity * sintheta\n            - costheta * temp\n            - self.viscous_friction_pole * theta_dot / max(self.polemass_length, 1e-8)\n        ) / (\n            self.length * (4.0 / 3.0 - self.masspole * costheta**2 / self.total_mass)\n        )\n\n        x_acc = temp - self.polemass_length * theta_acc * costheta / self.total_mass\n\n        x_dot = x_dot + self.dt * x_acc\n        x = x + self.dt * x_dot\n        theta_dot = theta_dot + self.dt * theta_acc\n        theta = theta + self.dt * theta_dot\n        theta = self.angle_normalize(theta)\n\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n        self.steps += 1\n\n        reward = self._reward(x, x_dot, theta, theta_dot, force)\n\n        terminated = bool(abs(x) > self.x_threshold)\n        truncated = bool(self.steps >= self.max_episode_steps)\n\n        self.last_u = force\n        self.cumulative_control_effort += force**2 * self.dt\n        self.cumulative_reward += reward\n\n        info = {\n            "x": x,\n            "x_dot": x_dot,\n            "theta": theta,\n            "theta_dot": theta_dot,\n            "u": force,\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n            "is_upright": abs(theta) < 0.15,\n        }\n\n        if self.render_mode == "human":\n            self.render()\n\n        return self._get_obs(), float(reward), terminated, truncated, info\n\n    def _reward(self, x: float, x_dot: float, theta: float, theta_dot: float, u: float) -> float:\n        """\n        Recompensa mejorada para swing-up y estabilización.\n        """\n\n        reward = 1.0 + math.cos(theta)\n\n        reward += 0.05\n\n        reward -= 0.20 * x**2\n        reward -= 0.01 * x_dot**2\n        reward -= 0.02 * theta_dot**2\n        reward -= 0.001 * u**2\n\n        if abs(theta) < 0.15:\n            reward += 3.0\n\n        if abs(x) > 0.80 * self.x_threshold:\n            reward -= 5.0 * (abs(x) - 0.80 * self.x_threshold)\n\n        if abs(x) > self.x_threshold:\n            reward -= 100.0\n\n        return reward\n\n    def render(self):\n        try:\n            import pygame\n            from pygame import gfxdraw\n        except ImportError as exc:\n            raise ImportError("Para renderizar instala pygame: pip install pygame") from exc\n\n        if self.screen is None:\n            pygame.init()\n            if self.render_mode == "human":\n                self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))\n            else:\n                self.screen = pygame.Surface((self.screen_width, self.screen_height))\n            self.clock = pygame.time.Clock()\n\n        assert self.state is not None\n        x, _, theta, _ = self.state\n\n        world_width = self.x_threshold * 2\n        scale = self.screen_width / world_width\n        cart_y = int(self.screen_height * 0.65)\n        cart_w = 60\n        cart_h = 30\n        pole_len = scale * (2 * self.length)\n\n        cart_x = int(self.screen_width / 2 + x * scale)\n\n        self.screen.fill((255, 255, 255))\n        import pygame\n\n        pygame.draw.line(\n            self.screen, (0, 0, 0),\n            (0, cart_y + cart_h // 2),\n            (self.screen_width, cart_y + cart_h // 2),\n            2\n        )\n\n        cart_rect = pygame.Rect(cart_x - cart_w // 2, cart_y - cart_h // 2, cart_w, cart_h)\n        pygame.draw.rect(self.screen, (30, 144, 255), cart_rect)\n        pygame.draw.rect(self.screen, (0, 0, 0), cart_rect, 2)\n\n        pivot = (cart_x, cart_y - cart_h // 2)\n        pole_x = pivot[0] + pole_len * math.sin(theta)\n        pole_y = pivot[1] - pole_len * math.cos(theta)\n\n        pygame.draw.line(self.screen, (220, 80, 20), pivot, (pole_x, pole_y), 6)\n        gfxdraw.filled_circle(self.screen, int(pole_x), int(pole_y), 10, (220, 80, 20))\n        gfxdraw.aacircle(self.screen, int(pole_x), int(pole_y), 10, (0, 0, 0))\n\n        font = pygame.font.SysFont("Arial", 18)\n        txt = font.render(f"theta={theta:.2f} rad | u={self.last_u:.2f} N", True, (0, 0, 0))\n        self.screen.blit(txt, (20, 20))\n\n        if self.render_mode == "human":\n            pygame.event.pump()\n            pygame.display.flip()\n            self.clock.tick(self.metadata["render_fps"])\n        else:\n            return np.transpose(\n                np.array(pygame.surfarray.pixels3d(self.screen)), axes=(1, 0, 2)\n            )\n\n    def close(self):\n        if self.screen is not None:\n            import pygame\n            pygame.display.quit()\n            pygame.quit()\n            self.screen = None\n            self.clock = None\n'
Path('envs').mkdir(exist_ok=True)
Path('envs/__init__.py').write_text('',encoding='utf-8')
Path('envs/cartpole_continuous_env.py').write_text(ORIGINAL_ENV,encoding='utf-8')
from envs.cartpole_continuous_env import CartPoleContinuousEnv
CFG=dict(dt=.02,steps=6000,force_max=10.0)
class CartPoleRewardR1(CartPoleContinuousEnv):
    def _reward(self,x,v,theta,omega,u):
        r=super()._reward(x,v,theta,omega,u)
        return r-(3.0 if abs(theta)<.15 and abs(omega)>=.50 else 0.)
class CartPoleRewardR2(CartPoleRewardR1):
    def _reward(self,x,v,theta,omega,u):
        r=super()._reward(x,v,theta,omega,u)
        local=abs(x)<.35 and abs(v)<.25 and abs(theta)<.15 and abs(omega)<.50
        return r+(3.0 if local else 0.)
def make_env(steps=6000,perturbation=0.):
    env=CartPoleRewardR2(max_episode_steps=steps,force_max=10.,dt=.02,
        x_threshold=2.4,noise_std=0.,viscous_friction_cart=0.,
        viscous_friction_pole=0.,random_initial_state=True,start_near_downward=True,
        external_impulse=float(perturbation),impulse_step=1000 if perturbation else None)
    return env
print('SHA-256 del entorno físico original:',hashlib.sha256(ORIGINAL_ENV.encode()).hexdigest())

In [ ]:
def discretizacion():
    env=make_env(10)
    z0=np.zeros(4,dtype=np.float64);eps=1e-5
    def f(z,u):
        env.state=np.asarray(z,dtype=np.float32).copy();env.steps=0
        _,_,_,_,_=env.step(np.array([u],dtype=np.float32))
        return env.state.astype(float)
    A=np.column_stack([(f(z0+eps*np.eye(4)[i],0.)-f(z0-eps*np.eye(4)[i],0.))/(2*eps) for i in range(4)])
    B=((f(z0,eps)-f(z0,-eps))/(2*eps)).reshape(4,1)
    env.close()
    return A,B
Ad,Bd=discretizacion()
Q=np.diag([1.,1.,30.,4.]);R=np.array([[0.5]])
P=solve_discrete_are(Ad,Bd,Q,R)
K=np.linalg.solve(Bd.T@P@Bd+R,Bd.T@P@Ad).reshape(1,4)
assert np.max(np.abs(np.linalg.eigvals(Ad-Bd@K)))<1
Ac=(Ad-np.eye(4))/.02;Bc=Bd/.02
C=np.array([[0.7,0.3,5.0,1.0]])
CB=float((C@Bc).item())
assert abs(CB)>1e-4
print('LQR K=',K.round(4),'polos discretos=',np.linalg.eigvals(Ad-Bd@K).round(4))
print('ISMC C B=',round(CB,4))

class ControlHibrido:
    def __init__(self,modelo,modo,theta_entrada=.20,omega_entrada=.75,theta_salida=None):
        assert modo in ('DDPG','DDPG-LQR','DDPG-ISMC')
        self.modelo=modelo;self.modo=modo
        self.te=theta_entrada;self.we=omega_entrada
        self.ts=theta_salida if theta_salida is not None else 1.5*theta_entrada
        self.local=False;self.z_entry=np.zeros(4);self.integral=0.
        self.switches=0
    def act(self,obs):
        z=np.asarray(obs,dtype=float).reshape(4)
        if self.modo=='DDPG':
            u,_=self.modelo.predict(np.asarray(obs,dtype=np.float32),deterministic=True)
            return float(np.clip(np.asarray(u).item(),-10,10))
        x,v,th,w=z
        if self.local:
            if abs(th)>self.ts or abs(w)>1.5*self.we or abs(x)>.85:
                self.local=False;self.switches+=1
        else:
            if abs(th)<self.te and abs(w)<self.we and abs(x)<.60 and abs(v)<1.0:
                self.local=True;self.z_entry=z.copy();self.integral=0.;self.switches+=1
        if not self.local:
            u,_=self.modelo.predict(np.asarray(obs,dtype=np.float32),deterministic=True)
            return float(np.clip(np.asarray(u).item(),-10,10))
        ulqr=-float((K@z).item())
        if self.modo=='DDPG-LQR': return float(np.clip(ulqr,-10,10))
        # Integral sliding surface: s=C(z-z_entrada)-∫C(A-BK)z dt.
        # Se actualiza en la misma frecuencia física de 50 Hz.
        self.integral+=float((C@(Ac-Bc@K)@z).item())*.02
        s=float((C@(z-self.z_entry)).item())-self.integral
        correction=(5.*s+0.65*np.tanh(s/.08))/CB
        return float(np.clip(ulqr-correction,-10,10))
print('Controladores definidos sin entrenamientos.')

## 4. Protocolo y controladores puros: ensayo de seguridad

In [ ]:
# Los checkpoints de la auditoría H1.1 quedan completamente excluidos.
H11_USED={(1102,'N5_CURRICULO'),(1104,'N5_CURRICULO'),(1104,'N6_RETENCION')}
POLICIES=sorted(set(models)-H11_USED)
assert len(POLICIES)==7, POLICIES
SEEDS=[8500,8501]
SCENARIOS=[
 dict(caso='sin_pulso',fuerza_N=0.,duracion_s=0.,inicio_s=20.),
 dict(caso='mas5_02s_t20',fuerza_N=5.,duracion_s=.2,inicio_s=20.),
 dict(caso='menos5_02s_t60',fuerza_N=-5.,duracion_s=.2,inicio_s=60.),
 dict(caso='mas10_02s_t20',fuerza_N=10.,duracion_s=.2,inicio_s=20.),
 dict(caso='mas5_1s_t60',fuerza_N=5.,duracion_s=1.,inicio_s=60.),
]
MODES=['DDPG','DDPG-LQR','DDPG-ISMC','LQR','ISMC']
GATE={'DDPG-LQR':.25,'DDPG-ISMC':.25}
DT=.02;STEPS=6000;STRICT_HOLD=50
assert STRICT_HOLD*DT==1.0
assert len(SEEDS)==2 and len(SCENARIOS)==5

class ControlLocalPuro:
    """LQR/ISMC aplicados en TODO estado. No tiene DDPG ni conmutación."""
    def __init__(self,modo):
        assert modo in ('LQR','ISMC');self.modo=modo
        self.local=True;self.switches=0;self.z_entry=np.zeros(4);self.integral=0.;self.initialized=False
    def act(self,obs):
        z=np.asarray(obs,dtype=float).reshape(4)
        if not self.initialized:
            self.z_entry=z.copy();self.initialized=True
        ulqr=-float((K@z).item())
        if self.modo=='LQR':return float(np.clip(ulqr,-10.,10.))
        self.integral+=float((C@(Ac-Bc@K)@z).item())*DT
        s=float((C@(z-self.z_entry)).item())-self.integral
        return float(np.clip(ulqr-(5.*s+.65*np.tanh(s/.08))/CB,-10.,10.))

def strict_ok(z):
    x,v,th,w=map(float,z)
    return abs(x)<=.20 and abs(v)<=.10 and abs(th)<=.05 and abs(w)<=.10

# Confrontación analítica: en ángulo pequeño, LQR estabiliza el modelo linealizado.
assert max(abs(np.linalg.eigvals(Ad-Bd@K)))<1
for mode in ('LQR','ISMC'):
    env=make_env(200,0.)
    obs,_=env.reset(seed=123)
    env.state=np.array([.01,.0,.02,.0],dtype=np.float32)
    obs=env._get_obs()
    ctrl=ControlLocalPuro(mode)
    theta_inicial=abs(float(env.state[2]))
    for _ in range(150):
        u=ctrl.act(obs)
        assert np.isfinite(u) and abs(u)<=10.+1e-6, (mode,u)
        obs,_,term,trunc,_=env.step(np.array([u],dtype=np.float32))
        assert np.all(np.isfinite(env.state)), (mode,'estado no finito',env.state)
        assert strict_ok(env.state), (mode,'fuera de limites estrictos',env.state)
        assert not term, (mode,'terminacion anticipada',env.state)
        if trunc:break
    # Prueba corta de estabilidad local, no demostracion de convergencia asintotica.
    assert abs(float(env.state[2]))<theta_inicial, (mode,'angulo no mejoro',env.state)
    print('Prueba local',mode,'estado final',env.state.round(6))
    env.close()
print('PRUEBAS OK: estabilidad local, saturación, semillas nuevas y separación de políticas.')

## 5. Simulador de evaluación estricta y control de perturbaciones

In [ ]:
def json_seguro(obj):
    if isinstance(obj,dict):return {str(k):json_seguro(v) for k,v in obj.items()}
    if isinstance(obj,(list,tuple)):return [json_seguro(v) for v in obj]
    if isinstance(obj,np.ndarray):return json_seguro(obj.tolist())
    if isinstance(obj,np.generic):return json_seguro(obj.item())
    if isinstance(obj,float):return obj if math.isfinite(obj) else None
    return obj
assert json.loads(json.dumps(json_seguro({'nan':np.nan,'inf':np.inf}),allow_nan=False))=={'nan':None,'inf':None}

def run_episode(model,mode,seed,scenario,initial='down',keep_trace=False):
    steps=STEPS if initial=='down' else 1500
    env=make_env(steps,0.)
    obs,_=env.reset(seed=int(seed))
    if initial=='upright':
        rng=np.random.default_rng(int(seed)+100000)
        # ABLACIÓN LOCAL. No usar como evidencia de éxito desde abajo.
        env.state=np.array([rng.uniform(-.03,.03),rng.uniform(-.02,.02),
                            rng.uniform(-.03,.03),rng.uniform(-.04,.04)],dtype=np.float32)
        obs=env._get_obs()
    else:
        assert abs(float(obs[2]))>2.9,'Arranque inferior obligatorio'
    z0=obs.copy()
    ctrl=ControlLocalPuro(mode) if mode in ('LQR','ISMC') else ControlHibrido(model,mode,theta_entrada=GATE.get(mode,.20),omega_entrada=.75)
    start=int(round(scenario['inicio_s']/DT));dur=int(round(scenario['duracion_s']/DT));end=start+dur
    assert start*DT==round(scenario['inicio_s'],4) and end<=steps
    ok_hist=[];traces=[];local_ticks=0;sat_ticks=0;energy=0.;longest=current=0
    entered_upright=None;first_local=None;exits=0;previous_local=False
    pulse_applied_steps=0;first_strict=None
    for t in range(steps):
        u=float(ctrl.act(obs))
        islocal=bool(ctrl.local)
        if islocal and not previous_local and first_local is None:first_local=t*DT
        if previous_local and not islocal:exits+=1
        previous_local=islocal;local_ticks+=int(islocal)
        sat_ticks+=int(abs(u)>=9.999);energy+=u*u*DT
        extra=float(scenario['fuerza_N']) if dur>0 and start<=t<end else 0.
        env.external_impulse=extra;env.impulse_step=t if extra else None
        if extra:pulse_applied_steps+=1
        obs,rew,term,trunc,info=env.step(np.array([u],dtype=np.float32))
        z=env.state.copy();ok=strict_ok(z)
        if abs(float(z[2]))<=.15 and entered_upright is None:entered_upright=(t+1)*DT
        if ok and first_strict is None:first_strict=(t+1)*DT
        current=current+1 if ok else 0;longest=max(longest,current)
        ok_hist.append(ok)
        if keep_trace:
            traces.append(dict(t_s=(t+1)*DT,x=float(z[0]),v=float(z[1]),theta=float(z[2]),omega=float(z[3]),
                               fuerza_actuador_N=u,fuerza_externa_N=extra,estricto=int(ok),local=int(islocal)))
        if term or trunc:break
    env.close();N=len(ok_hist)
    ok_arr=np.asarray(ok_hist,dtype=bool)
    stable_before=bool(dur and N>=start and start>=STRICT_HOLD and ok_arr[start-STRICT_HOLD:start].all())
    applied=(dur==0 or pulse_applied_steps==dur)
    recovery=np.nan
    if dur>0 and applied and N>=end+STRICT_HOLD:
        for j in range(end,N-STRICT_HOLD+1):
            if ok_arr[j:j+STRICT_HOLD].all():recovery=(j-end)*DT;break
    result=dict(arquitectura=mode,semilla_eval=int(seed),inicio=initial,caso=scenario['caso'],
        fuerza_N=float(scenario['fuerza_N']),duracion_s=float(scenario['duracion_s']),inicio_pulso_s=float(scenario['inicio_s']),
        pulso_completo=int(applied),estado_inicial_theta=float(z0[2]),pasos=N,supervivencia_s=N*DT,
        primer_angulo_vertical_s=entered_upright,primer_estricto_s=first_strict,
        estabilidad_max_s=longest*DT,ocupacion_estricta_pct=100*ok_arr.mean(),
        ocupacion_ultimos_30s_pct=100*ok_arr[max(0,N-1500):].mean(),
        esfuerzo_N2s=energy,saturacion_pct=100*sat_ticks/N,
        entrada_local_s=first_local,proporcion_local_pct=100*local_ticks/N,
        conmutaciones=int(ctrl.switches),salidas_local=int(exits),
        estable_antes_pulso=int(stable_before) if dur else None,
        recuperacion_s=recovery,recuperacion_valida=int(stable_before and applied and np.isfinite(recovery)) if dur else None,
        recuperacion_antes_15s=int(stable_before and applied and np.isfinite(recovery) and recovery<=15) if dur else None)
    return result,pd.DataFrame.from_records(traces)

# Prueba corta: las políticas puras no utilizan inferencia neuronal; arranque reproducible.
probe=SCENARIOS[0]
for mode in MODES:
    mm=None if mode in ('LQR','ISMC') else models[POLICIES[0]]
    r,_=run_episode(mm,mode,8500,probe,initial='down')
    assert 0<r['supervivencia_s']<=120 and np.isfinite(r['estabilidad_max_s'])
    assert abs(r['estado_inicial_theta'])>2.9
    if mode in ('LQR','ISMC'):assert r['proporcion_local_pct']==100 and r['conmutaciones']==0
print('PRUEBAS OK: los cinco modos funcionan desde abajo y las métricas son finitas.')

## 6. Ejecutar campaña final con reanudación en Drive

**No ajustar ganancias ni umbrales después de observar estas semillas.** Las arquitecturas LQR/ISMC puras se ejecutan una sola vez por condición, ya que son independientes del modelo DDPG. Presupuesto: **(7 × 3 + 2) × 5 × 2 = 230 episodios** de 120 s, más controles locales cortos.

In [ ]:
EP=OUT/'episodios';EP.mkdir(parents=True,exist_ok=True)
TR=OUT/'trazas';TR.mkdir(parents=True,exist_ok=True)
TASKS=[]
for key in POLICIES:
    for mode in ('DDPG','DDPG-LQR','DDPG-ISMC'):
        for cc in SCENARIOS:
            for seed in SEEDS:TASKS.append((key,mode,cc,seed))
for mode in ('LQR','ISMC'):
    for cc in SCENARIOS:
        for seed in SEEDS:TASKS.append((None,mode,cc,seed))
assert len(TASKS)==230
rows=[];started=time.perf_counter()
for i,(key,mode,cc,seed) in enumerate(TASKS,1):
    label=(f'{key[0]}_{key[1]}' if key is not None else 'SIN_DDPG')
    filename=f'{label}_{mode}_{cc["caso"]}_{seed}.json'
    path=EP/filename
    if path.exists():
        d=json.loads(path.read_text(encoding='utf8'))
        if (d.get('caso')==cc['caso'] and d.get('arquitectura')==mode and d.get('semilla_eval')==seed
            and d.get('train_seed')==(key[0] if key is not None else None)):
            rows.append(d);continue
    keep=(cc['caso']=='mas10_02s_t20' and seed==SEEDS[0] and (key==POLICIES[0] or key is None))
    d,trace=run_episode(models[key] if key is not None else None,mode,seed,cc,keep_trace=keep)
    d.update(train_seed=key[0] if key is not None else None,tratamiento=key[1] if key is not None else 'SIN_DDPG')
    if keep:trace.to_csv(TR/filename.replace('.json','.csv'),index=False)
    tmp=path.with_suffix('.partial')
    tmp.write_text(json.dumps(json_seguro(d),ensure_ascii=False,allow_nan=False,indent=2),encoding='utf8')
    tmp.replace(path)
    rows.append(d)
    if i%10==0:print(f'{i}/{len(TASKS)} — minutos {(time.perf_counter()-started)/60:.1f}',flush=True)
assert len(rows)==len(TASKS)
df=pd.DataFrame(rows)
assert df[['train_seed','tratamiento','arquitectura','caso','semilla_eval']].fillna('SIN_DDPG').duplicated().sum()==0
df.to_csv(OUT/'H12_episodios.csv',index=False)
print('COMPLETADO:',len(df),'episodios; resultados en',OUT)

## 7. Prueba de controles locales cerca del equilibrio (ablación separada)

In [ ]:
NEAR=[]
neutral=dict(caso='sin_pulso',fuerza_N=0.,duracion_s=0.,inicio_s=20.)
for mode in ('LQR','ISMC'):
    for seed in (8600,8601,8602):
        r,_=run_episode(None,mode,seed,neutral,initial='upright')
        NEAR.append(r)
near=pd.DataFrame(NEAR)
near.to_csv(OUT/'H12_local_cerca_vertical.csv',index=False)
print(near.groupby('arquitectura')[['estabilidad_max_s','ocupacion_estricta_pct']].mean().round(3))
print('Nota: iniciar vertical NO demuestra capacidad de swing-up desde abajo.')

## 8. Tablas, verificación de conmutaciones y conclusiones condicionales

In [ ]:
df=pd.read_csv(OUT/'H12_episodios.csv')
assert len(df)==230
summary=df.groupby('arquitectura').agg(episodios=('estabilidad_max_s','size'),
    estabilidad_s=('estabilidad_max_s','mean'),ocupacion_pct=('ocupacion_estricta_pct','mean'),
    llegada_vertical_pct=('primer_angulo_vertical_s',lambda a:100*a.notna().mean()),
    saturacion_pct=('saturacion_pct','mean'),esfuerzo_N2s=('esfuerzo_N2s','mean'),
    control_local_pct=('proporcion_local_pct','mean')).reset_index()
summary.to_csv(OUT/'H12_resumen_arquitecturas.csv',index=False)
print(summary.round(3).to_string(index=False))
print('\nRecuperación condicionada a estar estable antes de pulso:')
pert=df[(df['duracion_s']>0) & (df['estable_antes_pulso']==1) & (df['pulso_completo']==1)]
rec=pert.groupby('arquitectura').agg(casos_elegibles=('recuperacion_valida','size'),
     recuperaciones=('recuperacion_valida','sum'),antes_15s=('recuperacion_antes_15s','sum'),
     recuperacion_mediana_s=('recuperacion_s','median')).reset_index()
rec.to_csv(OUT/'H12_recuperacion.csv',index=False)
print(rec.to_string(index=False))
print('\nCruces de arquitectura y escenario:')
cross=df.pivot_table(index='caso',columns='arquitectura',values='estabilidad_max_s',aggfunc='mean')
cross.to_csv(OUT/'H12_por_escenario.csv')
print(cross.round(2).to_string())
# No confundir n=140 episodios de híbridos con 7 políticas independientes o con n=140 muestras independientes.
print('\nADVERTENCIA: los episodios comparten políticas de entrenamiento. Comparar diferencias emparejadas por política+semilla+escenario.')

## 9. Comparación emparejada y archivo final

In [ ]:
ddpg_df=df[df['arquitectura']=='DDPG'].set_index(['train_seed','tratamiento','caso','semilla_eval'])
paired=[]
for mode in ('DDPG-LQR','DDPG-ISMC'):
    h=df[df['arquitectura']==mode].set_index(['train_seed','tratamiento','caso','semilla_eval'])
    assert len(h)==len(ddpg_df)==70
    for idx in h.index:
        paired.append(dict(train_seed=idx[0],tratamiento=idx[1],caso=idx[2],semilla_eval=idx[3],
            arquitectura=mode,delta_estabilidad_s=float(h.loc[idx,'estabilidad_max_s']-ddpg_df.loc[idx,'estabilidad_max_s']),
            delta_ocupacion_pct=float(h.loc[idx,'ocupacion_estricta_pct']-ddpg_df.loc[idx,'ocupacion_estricta_pct'])))
pairs=pd.DataFrame(paired);pairs.to_csv(OUT/'H12_diferencias_emparejadas.csv',index=False)
print(pairs.groupby(['arquitectura','train_seed'])[['delta_estabilidad_s','delta_ocupacion_pct']].mean().round(2).to_string())
print('Políticas únicas H1.2:',len(POLICIES),'semillas de entrenamiento distintas:',len(set(k[0] for k in POLICIES)))

files_out=[OUT/x for x in ('H12_episodios.csv','H12_local_cerca_vertical.csv','H12_resumen_arquitecturas.csv',
  'H12_recuperacion.csv','H12_por_escenario.csv','H12_diferencias_emparejadas.csv')]
assert all(p.exists() for p in files_out)
zip_dest=Path('/content/H12_VALIDACION_HIBRIDOS_RESULTADOS.zip')
with zipfile.ZipFile(zip_dest,'w',compression=zipfile.ZIP_DEFLATED) as zz:
    for p in files_out:zz.write(p,p.name)
    for p in sorted(TR.glob('*.csv')):zz.write(p,'trazas/'+p.name)
    zz.writestr('PROTOCOLO.txt','H1.2; 7 checkpoints retenidos; 5 arquitecturas; 2 semillas 8500-8501; 5 escenarios; sin reentrenar. LQR/ISMC puros no dependen de modelo DDPG. Episodios completos conservados en Drive.\n')
assert zipfile.ZipFile(zip_dest).testzip() is None
print('EXPORTACIÓN OK',zip_dest,zip_dest.stat().st_size,'bytes')
files.download(str(zip_dest))

## Interpretación científica

1. **DDPG-LQR y DDPG-ISMC:** si superan al DDPG puro en políticas retenidas, la mejora corresponde al **sistema híbrido** y no a la optimización del aprendizaje neuronal.
2. **LQR e ISMC puros:** iniciar desde abajo permite probar si los controladores locales pueden realizar el *swing-up* por sí solos; iniciar cerca de arriba sirve para evaluar su estabilización local. **No comparar ambos estados iniciales como si fueran equivalentes.**
3. **Reproducibilidad:** las siete políticas ya fueron consideradas en H1, aunque no en H1.1, y provienen de cinco semillas de entrenamiento; la selección previa de checkpoints en N6.1 introduce condicionamiento por validación; no afirmar independencia estadística de episodios de una misma política.
4. **Perturbaciones:** registrar recuperación únicamente cuando existía estabilidad antes del pulso; los episodios no elegibles no son fallos de recuperación.
5. **Sin modificación de parámetros:** si los resultados no son consistentes, informar esa limitación. No ajustar ganancias y volver a reutilizar el mismo conjunto final como si fuera evaluación independiente.